# Stage 1c — Checking the meanings we only guessed

**Purpose.** In stage 1b some column meanings were only **inferred [I]** from names and examples. A guess can be wrong, so here we test each one with code. The idea is simple: if a column means what we think, it must behave in a way we can predict, and we check that it does.

| Column | What we think it is | How we test it | What we expect if we are right |
|---|---|---|---|
| `related` | Links from an anime to other titles (sequel, prequel, adaptation...) | Parse the text into a table; check that a "Sequel" link is mirrored by a "Prequel" link | Most sequel links have a matching prequel link, and the linked ids exist in the anime table |
| `popularity` | A **rank** by number of members (1 = most popular) | Rank correlation with `members`, and with `score` as a control | Strongly **negative** with `members` (more members, smaller number), and clearly **weaker** with `score`, because being popular is not the same as being well rated |
| `rank` | A rank by `score` (1 = best score) | Rank correlation with `score` | Strongly **negative** |
| `members`, `scored_by`, `favorites` | Counts of users | Logical limits, and comparison with our own counts from the list table | `scored_by` and `favorites` never exceed `members`; our counts are proportional to the published ones |
| `score` | Average of the users' scores | Compare with the mean of the scores in our list table | Strong positive correlation |
| `episodes` | Number of episodes | Look at it by `type` | Movies have 1 episode; 0 may mean unknown |
| `stats_rewatched` | Something about rewatching, exact count unknown: anime rewatched, times rewatched, or episodes rewatched | Compare with the size of the user's list and with `stats_episodes`; compare with our own counts | Each reading makes a different prediction (see part 4) |
| `stats_episodes`, `stats_mean_score` | Total episodes watched, and the user's mean score | Recompute both from the user's rows in the list table | Close to the recomputed values |

**Important limit.** The list table covers only the users the author crawled (about 283,000), not everybody on MyAnimeList. So our own counts will be **smaller** than the published ones. We therefore look for strong agreement in the *order* of the anime (rank correlation) and a *stable ratio*, not for equal numbers.

**How to run on Kaggle.** Parts 1 and 2 take seconds. Part 3 reads two columns of the 80M-row list table (a minute or two); set `RUN_LISTS = False` in the setup cell to skip it.

In [1]:
from pathlib import Path
import ast
import sys

import pandas as pd

# Works both on Kaggle and inside the repo (with the data placed in data/raw/)
KAGGLE_INPUT = Path("/kaggle/input")
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "environment.yml").exists()), None)
if KAGGLE_INPUT.exists():
    DATA_DIR = KAGGLE_INPUT
elif ROOT is not None:
    DATA_DIR = ROOT / "data" / "raw"
else:
    raise RuntimeError("Data not found: run on Kaggle or inside the repository.")

paths = {p.name: p for p in DATA_DIR.rglob("*.csv")}
RUN_LISTS = True  # False skips part 3, which reads 80M rows
na_kwargs = dict(keep_default_na=False, na_values=[""])  # only an empty cell is missing

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)

anime = pd.read_csv(paths["AnimeList.csv"], **na_kwargs)
print("Python:", sys.version.split()[0], "| pandas:", pd.__version__, "| anime:", anime.shape)

Python: 3.13.18 | pandas: 2.3.3 | anime: (14478, 31)


## 1. What is inside `related`?

Earlier the profile table cut every example at 45 characters, so only the first words were visible. Here we print three complete cells.

In [2]:
for anime_id in anime.anime_id.head(3):
    print(anime_id, anime.loc[anime.anime_id == anime_id, "title"].iloc[0])
    print(anime.loc[anime.anime_id == anime_id, "related"].iloc[0], "\n")

11013 Inu x Boku SS
{'Adaptation': [{'mal_id': 17207, 'type': 'manga', 'url': 'https://myanimelist.net/manga/17207/Inu_x_Boku_SS', 'title': 'Inu x Boku SS'}], 'Sequel': [{'mal_id': 13403, 'type': 'anime', 'url': 'https://myanimelist.net/anime/13403/Inu_x_Boku_SS_Special', 'title': 'Inu x Boku SS Special'}]} 

2104 Seto no Hanayome
{'Adaptation': [{'mal_id': 759, 'type': 'manga', 'url': 'https://myanimelist.net/manga/759/Seto_no_Hanayome', 'title': 'Seto no Hanayome'}], 'Sequel': [{'mal_id': 4163, 'type': 'anime', 'url': 'https://myanimelist.net/anime/4163/Seto_no_Hanayome_OVA', 'title': 'Seto no Hanayome OVA'}], 'Side story': [{'mal_id': 5199, 'type': 'anime', 'url': 'https://myanimelist.net/anime/5199/Seto_no_Hanayome_Fan_Disc', 'title': 'Seto no Hanayome Fan Disc'}], 'Alternative version': [{'mal_id': 28385, 'type': 'anime', 'url': 'https://myanimelist.net/anime/28385/Seto_no_Hanayome_Picture_Drama', 'title': 'Seto no Hanayome Picture Drama'}]} 

5262 Shugo Chara!! Doki
{'Adaptation'

The cell is the **text of a Python dictionary**. The keys are kinds of relation (for example `Adaptation`, `Sequel`), and each key holds a list of related titles. One cell can therefore describe several links, which is the "many values in one cell" problem from the tidy-data lesson. The code below turns it into a table with **one row per link**: `anime_id`, `relation`, `related_id`, `related_type`.

In [3]:
def parse_related(text):
    """Turn the text of one `related` cell into a list of (relation, related_id, related_type)."""
    try:
        value = ast.literal_eval(text)
    except (ValueError, SyntaxError):
        return None  # the text cannot be parsed
    if value == []:
        return []  # an empty list means "no relation"
    if not isinstance(value, dict):
        return None
    rows = []
    for relation, items in value.items():
        for item in items:
            if isinstance(item, dict):
                rows.append((relation, item.get("mal_id"), item.get("type")))
    return rows


parsed = anime["related"].map(parse_related)
print("cells that could not be parsed:", int(parsed.isna().sum()))
print("anime whose cell holds no link:", int(parsed.map(lambda r: r == []).sum()))
print("what the unparsable cells look like:")
print(anime.loc[parsed.isna(), "related"].str.slice(0, 60).value_counts().head(5).to_string())

links = pd.DataFrame({"anime_id": anime.anime_id, "link": parsed}).dropna(subset=["link"]).explode("link")
links = links.dropna(subset=["link"])
link_cols = pd.DataFrame(links["link"].tolist(), index=links.index,
                         columns=["relation", "related_id", "related_type"])
relations = pd.concat([links[["anime_id"]], link_cols], axis=1).reset_index(drop=True)
relations["related_id"] = pd.to_numeric(relations["related_id"], errors="coerce")

print(f"\nlink rows: {len(relations):,} | anime with at least one link: {relations.anime_id.nunique():,}")
print("\nrelation types:")
print(relations["relation"].value_counts().to_string())
print("\nrelated_type (what the linked title is):")
print(relations["related_type"].value_counts(dropna=False).to_string())

cells that could not be parsed: 0
anime whose cell holds no link: 4515
what the unparsable cells look like:
Series([], )

link rows: 20,611 | anime with at least one link: 9,963

relation types:
relation
Adaptation             4758
Other                  2996
Sequel                 2550
Prequel                2535
Parent story           1923
Side story             1700
Alternative version    1631
Alternative setting     715
Spin-off                573
Full story              437
Summary                 422
Character               371

related_type (what the linked title is):
related_type
anime    15853
manga     4758


In [4]:
# Do the linked ids exist in the anime table? (only links to anime, not to manga)
anime_links = relations[relations["related_type"] == "anime"]
found = anime_links["related_id"].isin(anime["anime_id"])
print(f"links to anime: {len(anime_links):,} | target id found in the anime table: {found.mean():.1%}")

# Is a Sequel link mirrored by a Prequel link in the other direction?
sequel = relations.loc[relations["relation"] == "Sequel", ["anime_id", "related_id"]]
prequel = (relations.loc[relations["relation"] == "Prequel", ["anime_id", "related_id"]]
           .rename(columns={"anime_id": "related_id", "related_id": "anime_id"}))
check = sequel.merge(prequel, on=["anime_id", "related_id"], how="left", indicator=True)
print(f"Sequel links: {len(check):,} | mirrored by a Prequel link: {(check['_merge'] == 'both').mean():.1%}")

links to anime: 15,853 | target id found in the anime table: 99.5%
Sequel links: 2,550 | mirrored by a Prequel link: 96.5%


### Observations for part 1 (`related`)

**What a cell contains.** The text of a Python dictionary. Each key is a kind of relation and holds a list of linked titles; each linked title has four fields: `mal_id` (its id), `type` (`anime` or `manga`), `url` and `title`. Example: Inu x Boku SS has an `Adaptation` of manga 17207 and a `Sequel`, anime 13403 (`Inu x Boku SS Special`).

**Parsing.** 9,963 anime have at least one link and together they hold 20,611 links. The other **4,515 anime (31.2%) have no link at all**, and 9,963 + 4,515 = 14,478, so every anime is in one group or the other. In the first run our parser rejected these 4,515 cells because they were not dictionaries. We then changed the parser to read an empty list `[]` as "no relation". After that change, **0 cells are unparsable**, the 4,515 are counted as "cell holds no link", and the cell that prints unparsable texts printed nothing. So these cells are empty lists, not damaged text and not missing values. The earlier first run also showed that none of them was an empty dictionary `{}` (that count was 0). An anime without a relation is therefore written as `[]` in this file.

**Relation types (12):** Adaptation 4,758, Other 2,996, Sequel 2,550, Prequel 2,535, Parent story 1,923, Side story 1,700, Alternative version 1,631, Alternative setting 715, Spin-off 573, Full story 437, Summary 422, Character 371. There are 4,758 Adaptation links and exactly 4,758 links to a manga, which suggests that Adaptation links are the ones that point to a manga (we did not cross-check row by row); the other 15,853 links point to anime.

**Checks passed.**
- **99.5%** of the links to anime point to an `anime_id` that exists in the anime table. The remaining 0.5% (about 79 links) point to titles not in this dataset.
- **96.5%** of the 2,550 Sequel links are mirrored by a Prequel link in the other direction, and the two relation counts are close (2,550 vs 2,535). So `Sequel` and `Prequel` mean what their names say, and `related` is confirmed **[V]**.

**What this gives us.** A usable table of sequel links (2,550) that can answer sequel questions, and we know that the 4,515 anime without a link are standalone titles, not missing data. For a question that compares a season with the next one, we will also need users who rated both, so the real sample will be smaller than 2,550.

## 2. Do the audience columns behave like ranks and counts?

`popularity` and `rank` are expected to be ranks: **1 is the best**. A rank goes *down* when the thing it measures goes *up*, so the rank correlation with `members` (for `popularity`) and `score` (for `rank`) should be close to **−1**. Rank correlation (Spearman) compares the order of the values and ignores their size, which is what we want for a rank.

**A control test.** A correlation of −0.99 with `members` shows that `popularity` follows `members`, but it does not show that `popularity` is *only* about `members`. So we also correlate each rank with the *other* measure (`popularity` with `score`, and `rank` with `members`). If our reading is right, these cross correlations should be clearly weaker than the matching ones: an anime can have many members and a mediocre score, or few members and a high score. If they were just as strong, the two ranks would carry almost the same information and the matching correlation would prove less than it seems.

In [5]:
def spearman(a, b):
    """Rank correlation: correlate the positions of the values, not the values themselves."""
    return a.rank().corr(b.rank())


print("popularity vs members:", round(spearman(anime["popularity"], anime["members"]), 3),
      "(close to -1 means popularity is a rank by members)")
ranked = anime.dropna(subset=["rank"])
print("rank vs score        :", round(spearman(ranked["rank"], ranked["score"]), 3),
      "(close to -1 means rank is a rank by score)")

# Control: each rank against the OTHER measure. Only anime with a real value in both columns.
cross = anime[(anime["popularity"] > 0) & (anime["score"] > 0)]
print("\ncontrol (should be clearly weaker than the matching correlations above):")
print("  popularity vs score  :", round(spearman(cross["popularity"], cross["score"]), 3),
      f"({len(cross):,} anime with popularity > 0 and score > 0)")
cross_rank = ranked[ranked["members"] > 0]
print("  rank vs members      :", round(spearman(cross_rank["rank"], cross_rank["members"]), 3),
      f"({len(cross_rank):,} anime with a rank and members > 0)")

print("\nlogical limits that must hold if the meanings are right:")
print("  scored_by > members :", int((anime["scored_by"] > anime["members"]).sum()), "anime")
print("  favorites > members :", int((anime["favorites"] > anime["members"]).sum()), "anime")
print("  score outside 0-10  :", int((~anime["score"].between(0, 10)).sum()), "anime")
print("  popularity is 0 or negative:", int((anime["popularity"] <= 0).sum()), "anime")
print("  popularity values unique:", anime["popularity"].is_unique,
      "| distinct:", anime["popularity"].nunique(), "of", len(anime))

popularity vs members: -0.996 (close to -1 means popularity is a rank by members)
rank vs score        : -0.855 (close to -1 means rank is a rank by score)

control (should be clearly weaker than the matching correlations above):
  popularity vs score  : -0.737 (14,122 anime with popularity > 0 and score > 0)
  rank vs members      : -0.862 (12,900 anime with a rank and members > 0)

logical limits that must hold if the meanings are right:
  scored_by > members : 0 anime
  favorites > members : 1 anime
  score outside 0-10  : 0 anime
  popularity is 0 or negative: 9 anime
  popularity values unique: False | distinct: 11759 of 14478


In [6]:
# episodes by type: do the numbers look like episode counts?
print(anime.groupby("type")["episodes"].agg(
    anime="size", zero=lambda s: int((s == 0).sum()), median="median", max="max").to_string())

         anime  zero  median   max
type                              
Movie     2624     3     1.0   100
Music      849     4     1.0    30
ONA       1144   118     1.0    84
OVA       3612    60     1.0   110
Special   1948    17     1.0    59
TV        4271   272    22.0  1818
Unknown     30    30     0.0     0


### Observations for part 2 (ranks, counts, episodes)

**`popularity` is a rank by members.** Spearman correlation with `members` is **−0.996**, so more members means a smaller number, and 1 is the most popular. As a control we also correlated it with `score` (14,122 anime with both): **−0.737**. That is clearly weaker, so `popularity` follows `members` much more than it follows `score`. It is still fairly strong, which shows that popular anime also tend to have better scores. It is not perfect:
- **9 anime have a popularity of 0 or less**, which is a placeholder, not a rank.
- The values are **not unique**: 11,759 distinct values for 14,478 anime, so about 2,700 anime share a number with another one. A true ranking would not have ties, so some values may come from different moments of the crawl. We do not know.

**`rank` follows `score`, but loosely, and the control does not separate it from `members`.** The correlation with `score` is **−0.855**, weaker than for `popularity`. The control, `rank` against `members` (12,900 anime), is **−0.862**, which is just as strong and even slightly higher. For `popularity` the control worked (−0.996 against −0.737), but for `rank` it did not: we cannot say that `rank` is ordered by `score` rather than by `members`. Because popular anime tend to have better scores (the −0.737 above), the two are hard to pull apart with correlations. So `rank` is **not simply the order of `score`**. A plausible reason is that the site ranks with a formula that also uses the number of voters, but the data does not show that. It remains a hypothesis. For analysis, use `score` when we want the score and `rank` only as a rank, and do not assume one can be recomputed from the other.

**Logical limits.** All hold except for a few rows:
- `scored_by > members`: 0 anime. `score` outside 0–10: 0 anime.
- **`favorites > members` for 1 anime**, which should not be possible. One row to look at, probably an error.

**`episodes` means the number of episodes, and 0 means unknown.**

| Type | Anime | `episodes = 0` | Median | Maximum |
|---|---|---|---|---|
| TV | 4,271 | 272 | 22 | 1,818 |
| OVA | 3,612 | 60 | 1 | 110 |
| Special | 1,948 | 17 | 1 | 59 |
| Movie | 2,624 | 3 | 1 | 100 |
| ONA | 1,144 | 118 | 1 | 84 |
| Music | 849 | 4 | 1 | 30 |
| Unknown | 30 | 30 | 0 | 0 |

Movies have a median of 1 episode, as expected. All 30 anime of type `Unknown` have 0 episodes, and 272 TV and 118 ONA anime have 0, which fits "unknown or still airing" (351 anime are currently airing). A Movie with 100 episodes is suspicious and worth a look.

## 3. Compare the published numbers with our own counts

From the list table we can recompute, for every anime, how many of *our* users have it on their list (compare with `members`), how many of them scored it (compare with `scored_by`), and the mean of their scores (compare with `score`). The numbers will be smaller than the published ones because we only have a sample of the users, so we look at the **rank correlation** and at the **ratio** between our count and the published count.

In [7]:
if RUN_LISTS:
    lists = pd.read_csv(paths["UserAnimeList.csv"], usecols=["anime_id", "my_score"],
                        dtype={"anime_id": "int32", "my_score": "int8"})
    rated = lists[lists["my_score"] > 0]
    ours = pd.DataFrame({
        "on_list_sample": lists.groupby("anime_id").size(),
        "rated_sample": rated.groupby("anime_id").size(),
        "mean_score_sample": rated.groupby("anime_id")["my_score"].mean(),
    })
    both = anime.set_index("anime_id")[["members", "scored_by", "score"]].join(ours, how="left")
    both[["on_list_sample", "rated_sample"]] = both[["on_list_sample", "rated_sample"]].fillna(0)
    print(f"list rows: {len(lists):,} | anime: {len(both):,} | with at least one list row:",
          int((both['on_list_sample'] > 0).sum()))

    def compare(published, sample, label):
        ok = both[(both[published] > 0) & (both[sample] > 0)]
        ratio = ok[sample] / ok[published]
        print(f"{label:34s} anime compared: {len(ok):>6,} | Spearman: {spearman(ok[published], ok[sample]):.3f}"
              f" | median ratio: {ratio.median():.3f} | middle 90% of ratios: {ratio.quantile(.05):.3f} to {ratio.quantile(.95):.3f}")

    compare("members", "on_list_sample", "members vs users with it on list")
    compare("scored_by", "rated_sample", "scored_by vs users who scored it")

    enough = both[(both["rated_sample"] >= 30) & (both["score"] > 0)]
    diff = enough["mean_score_sample"] - enough["score"]
    print(f"{'score vs mean score in sample':34s} anime compared: {len(enough):>6,} | Pearson: "
          f"{enough['score'].corr(enough['mean_score_sample']):.3f} | mean difference: {diff.mean():+.3f}"
          f" | middle 90%: {diff.quantile(.05):+.2f} to {diff.quantile(.95):+.2f}")
else:
    print("Skipped (RUN_LISTS = False)")

list rows: 80,076,112 | anime: 14,478 | with at least one list row: 14478
members vs users with it on list   anime compared: 14,474 | Spearman: 0.994 | median ratio: 0.390 | middle 90% of ratios: 0.205 to 0.551
scored_by vs users who scored it   anime compared: 14,043 | Spearman: 0.995 | median ratio: 0.500 | middle 90% of ratios: 0.240 to 0.732
score vs mean score in sample      anime compared: 10,732 | Pearson: 0.976 | mean difference: -0.170 | middle 90%: -0.69 to +0.09


### Observations for part 3 (published numbers against our own counts)

Our list table holds about 283,000 users, so it is a **subset** of the users behind the published numbers. The comparison therefore asks whether the *order* and the *ratio* agree.

| Published column | Our own count | Anime compared | Spearman | Median ratio (ours ÷ published) | Middle 90% of ratios |
|---|---|---|---|---|---|
| `members` | Users with the anime on their list | 14,474 | **0.994** | 0.390 | 0.205 to 0.551 |
| `scored_by` | Users who scored the anime | 14,043 | **0.995** | 0.500 | 0.240 to 0.732 |

- **Both columns mean what we thought [V].** The order of the anime is almost identical (0.994 and 0.995).
- **The ratio is not constant.** For a typical anime our data holds 39% of the published members, but for the middle 90% of anime it ranges from 21% to 55%. So the crawled users do not mirror the full community: some anime are over-represented in our sample and others under-represented. Any popularity-based result needs this caveat.
- **Our users score more often than the published numbers imply.** The median ratio is higher for `scored_by` (0.50) than for `members` (0.39). This is a hint, not a proof, that the crawled users rate their anime more often than the average user.

**`score` is the average of the users' scores [V].** For 10,732 anime that at least 30 of our users scored, the Pearson correlation between the published `score` and the mean of our users' scores is **0.976**. On average our users score **0.17 lower** than the published average (middle 90% of anime: from −0.69 to +0.09), so published scores look slightly higher than what our sample says.

## 4. What do the per-user statistics count?

`stats_rewatched` can be read in three ways, and each reading predicts something different:

| Reading | Prediction we can test |
|---|---|
| (a) Number of **anime** the user has rewatched | Never larger than the number of anime on the user's list |
| (b) Total number of **times** the user rewatched anime | Can be larger than the list size, because one anime can be rewatched many times |
| (c) Number of rewatched **episodes** | Can be much larger than the list size, and cannot exceed `stats_episodes` |

The checks below only use the user table. Then the list table is used to recompute `stats_episodes` and `stats_mean_score` for every user. Even if the numbers fit one reading, the exact definition is MyAnimeList's, so what we can claim is "behaves like ...", not "is ...".

In [8]:
users = pd.read_csv(paths["UserList.csv"], **na_kwargs)
count_cols = ["user_watching", "user_completed", "user_onhold", "user_dropped", "user_plantowatch"]
users["list_size"] = users[count_cols].sum(axis=1)  # anime on the list, from the user table itself
known = users.dropna(subset=["stats_rewatched"])

print("users:", len(users), "| with stats_rewatched:", len(known))
print("whole numbers only:", bool((known["stats_rewatched"] % 1 == 0).all()),
      "| negative values:", int((known["stats_rewatched"] < 0).sum()))
print(known["stats_rewatched"].describe(percentiles=[.5, .9, .99, .999]).round(1).to_string())
print("share of users with stats_rewatched = 0:", f"{(known['stats_rewatched'] == 0).mean():.1%}")

print("\nstats_rewatched > list size   :", int((known["stats_rewatched"] > known["list_size"]).sum()),
      "users   (0 would fit reading (a))")
print("stats_rewatched > stats_episodes:", int((known["stats_rewatched"] > known["stats_episodes"]).sum()),
      "users   (0 would fit reading (c))")

print("\nthe 10 largest values, next to the size of the user's list and their stats_episodes:")
print(known.nlargest(10, "stats_rewatched")[
    ["username", "stats_rewatched", "list_size", "user_completed", "stats_episodes"]].to_string(index=False))

users: 302675 | with stats_rewatched: 302546
whole numbers only: True | negative values: 0
count    302546.0
mean         10.4
std          44.2
min           0.0
50%           0.0
90%          26.0
99%         149.0
99.9%       461.5
max        9404.0
share of users with stats_rewatched = 0: 58.9%

stats_rewatched > list size   : 5086 users   (0 would fit reading (a))
stats_rewatched > stats_episodes: 52 users   (0 would fit reading (c))

the 10 largest values, next to the size of the user's list and their stats_episodes:
       username  stats_rewatched  list_size  user_completed  stats_episodes
          Luica           9404.0       1466            1389        139337.0
DestinyofShadow           3998.0        470             457        747731.0
      StopWatch           3222.0       3866            2034         96677.0
    Disillusion           2791.0       3928            1290         31780.0
       Infernos           2607.0       1820            1656         56121.0
       Afezeria

In [9]:
if RUN_LISTS:
    cols = {"username": "category", "my_score": "int8", "my_rewatching": "float32", "my_watched_episodes": "int32"}
    user_lists = pd.read_csv(paths["UserAnimeList.csv"], usecols=list(cols), dtype=cols, **na_kwargs)
    user_lists["rewatching_now"] = user_lists["my_rewatching"] == 1
    user_lists["score_if_rated"] = user_lists["my_score"].where(user_lists["my_score"] > 0)
    by_user = user_lists.groupby("username", observed=True)
    ours = pd.DataFrame({
        "rows": by_user.size(),
        "rewatching_rows": by_user["rewatching_now"].sum(),
        "episodes_sum": by_user["my_watched_episodes"].sum(),
        "mean_score_ours": by_user["score_if_rated"].mean(),
    })
    ours.index = ours.index.astype(object)
    joined = (users.set_index("username")[["stats_rewatched", "stats_episodes", "stats_mean_score", "list_size"]]
              .join(ours, how="inner").dropna(subset=["stats_rewatched"]))
    print(f"users with list rows and statistics: {len(joined):,}")

    def agreement(published, recomputed, label):
        ok = joined[(joined[published] > 0) & (joined[recomputed] > 0)]
        ratio = ok[recomputed] / ok[published]
        exact = (joined[published] == joined[recomputed]).mean()
        print(f"{label:46s} users compared: {len(ok):>7,} | Spearman: {spearman(ok[published], ok[recomputed]):.3f}"
              f" | exact equal (all users): {exact:.1%} | median ratio ours/published: {ratio.median():.3f}")

    agreement("stats_episodes", "episodes_sum", "stats_episodes vs sum of my_watched_episodes")
    agreement("stats_rewatched", "rewatching_rows", "stats_rewatched vs rows marked rewatching")
    agreement("stats_mean_score", "mean_score_ours", "stats_mean_score vs mean of our scores")
    diff = (joined["mean_score_ours"] - joined["stats_mean_score"]).dropna()
    print(f"mean score difference (ours - published): mean {diff.mean():+.3f}, "
          f"middle 90% from {diff.quantile(.05):+.2f} to {diff.quantile(.95):+.2f}, "
          f"within 0.05: {(diff.abs() <= 0.05).mean():.1%}")
else:
    print("Skipped (RUN_LISTS = False)")

users with list rows and statistics: 282,918
stats_episodes vs sum of my_watched_episodes   users compared: 280,593 | Spearman: 0.976 | exact equal (all users): 50.8% | median ratio ours/published: 1.000
stats_rewatched vs rows marked rewatching      users compared:  14,686 | Spearman: 0.140 | exact equal (all users): 54.5% | median ratio ours/published: 0.125
stats_mean_score vs mean of our scores         users compared: 271,548 | Spearman: 1.000 | exact equal (all users): 14.3% | median ratio ours/published: 1.000
mean score difference (ours - published): mean +0.000, middle 90% from -0.00 to +0.00, within 0.05: 99.7%


### Observations for part 4 (per-user statistics)

**`stats_mean_score` is the user's mean score [V].** It agrees with the mean of the user's own scores (zeros excluded) within 0.05 for **99.7%** of the 271,548 users, the rank correlation is 1.000 and the mean difference is 0.000. Exact equality is only 14.3%, probably because the published value is rounded to 2 decimals and ours is not (not checked).

**`stats_episodes` is the total of episodes watched [V], but it contains impossible values.**
- The median ratio between our sum of `my_watched_episodes` and the published value is exactly **1.000**, the rank correlation is **0.976**, and the two are equal for 50.8% of the users. So it measures the same thing, but a large part of the users differ somewhat, and we do not know why.
- Some published values cannot be real. `Kitayama_Naoki` has **10,989,789** episodes, and `DestinyofShadow` has 747,731 episodes with only 470 anime on the list. These need a cap or removal before any use of `stats_episodes`.

**`stats_rewatched` is a count of rewatching, but not the number of anime rewatched.**
- Values are whole numbers, never negative. 58.9% of users have 0, the 99th percentile is 149, the maximum is 9,404.
- It is larger than the user's list size for **5,086 users**. A user cannot have rewatched more anime than they have, so reading (a), *number of anime rewatched*, is ruled out. Example: `Luica` has 9,404 with a list of 1,466 anime, and `DestinyofShadow` has 3,998 with a list of 470.
- It is larger than `stats_episodes` for only **52 users**. So reading (c), *episodes rewatched*, is possible for almost everybody, and reading (b), *times rewatched*, fits too. The exceptions could be errors.
- It does not match the rows marked as rewatching: the rank correlation is only **0.140** and our count is about 12% of the published value for the median user (0.125). `my_rewatching = 1` is a snapshot of who is rewatching now, while `stats_rewatched` accumulates over time.
- **Verdict:** we cannot tell (b) from (c) with the data we have, because the list table holds no per-anime rewatch count. The column stays **[I]**: "behaves like a total count of rewatches".

## After running

For every column whose behaviour matched the expectation in the table at the top, the tag in the stage 1b dictionary was changed from **[I]** to **[V]** with the evidence next to it.

| Column | Verdict | Evidence |
|---|---|---|
| `related` | Confirmed [V] | 99.5% of linked anime ids exist; 96.5% of Sequel links are mirrored by a Prequel link; the 4,515 anime without a link are written as `[]` |
| `popularity` | Confirmed [V], with exceptions | Rank correlation −0.996 with `members` against −0.737 with `score` (control); 9 anime have a value of 0 or less; values are not unique |
| `rank` | Not the plain order of `score` | Rank correlation −0.855 with `score` but −0.862 with `members`, so the control does not separate them |
| `members`, `scored_by` | Confirmed [V] | Rank correlation 0.994 and 0.995 with our own counts |
| `score` | Confirmed [V] | Correlation 0.976 with the mean of our users' scores, ours 0.17 lower |
| `favorites` | Still inferred [I] | Cannot be recomputed from our tables; larger than `members` for 1 anime |
| `episodes` | Confirmed [V]; 0 means unknown | Movies have a median of 1; 0 appears for all 30 `Unknown` type anime |
| `stats_mean_score` | Confirmed [V] | Within 0.05 of our recomputed mean for 99.7% of the users |
| `stats_episodes` | Confirmed [V], contains impossible values | Median ratio 1.000 with our sum; values up to 10,989,789 |
| `stats_rewatched` | Still inferred [I] | Larger than the list size for 5,086 users, so not "anime rewatched"; (b) and (c) cannot be separated |

**Open items for the next run:** look at the anime with `favorites > members` and the Movie with 100 episodes, and cap or remove impossible values of `stats_episodes`.